# 3. Hashing: Turning Data into a Fingerprint

This notebook covers **cryptographic hash functions**: tools that turn any input
into a fixed-size "fingerprint," in a way that is deterministic, one-way, and
extremely sensitive to small changes.

By the end of this notebook you should be able to answer:

- What three properties make a hash function useful for security?
- Why do systems store *hashes* of passwords instead of the passwords themselves?
- Why is a plain hash still not quite enough for password storage, and what's
  typically added?

Hashing isn't the same thing as encryption. There's no key, and there's no way to
"decrypt" a hash back into the original input. That irreversibility is the entire
point.

## 3.1 A basic hash, with SHA-256


In [1]:
import hashlib

def sha256_hex(data: str) -> str:
    return hashlib.sha256(data.encode()).hexdigest()

print(sha256_hex("hello world"))
print(sha256_hex("Hello world"))   # capital H
print(sha256_hex("hello world "))  # trailing space


b94d27b9934d3e08a52e52d7da7dabfac484efe37a5380ee9088f7ace2efcde9
64ec88ca00b268e5ba1a35678a1b5316d212f4f366b2477232534a8aeca37f3c
9e1b04359ce9f650852b7f468eb9cbaa8198d8678a7ddce5caaeb123b76439ff


Notice that changing a single character produces a completely different-looking
output. There's no visible relationship between similar inputs and their hashes.
This is called the **avalanche effect**, and it's deliberate: it means a hash can't
be used to guess anything about nearby inputs.

## 3.2 The three properties that matter

1. **Deterministic**: the same input always produces the same hash.
2. **One-way**: given a hash, there's no practical way to work backward to the
   original input (other than guessing inputs and hashing each one, which is
   exactly what an attacker with a stolen password database has to resort to).
3. **Collision-resistant**: it's computationally infeasible to find two different
   inputs that produce the same hash.

Let's confirm property 1 concretely:

In [2]:
a = sha256_hex("password123")
b = sha256_hex("password123")
print(a == b, "-", a)


True - ef92b778bafe771e89245b89ecbc08a44a4e166c06659911881f383d4473e94f


## 3.3 Why login systems store hashes, not passwords

If a database stores your password in plaintext and that database is ever leaked
(via a bug, an insider, or a breach), every user's real password is exposed
immediately. Since people reuse passwords, that damage spreads to other
sites too.

Instead, a well-built login system stores only `hash(password)`. When you log in,
it hashes what you typed and compares the two hashes. It never needs to know or
store your actual password to do this.

In [ ]:
# A tiny toy "user database" - never do this with plain SHA-256 in a real system,
# for reasons explained just below.
fake_user_db = {
    "alice": sha256_hex("correct-horse-battery-staple"),
}

def check_login(username: str, attempted_password: str) -> bool:
    stored_hash = fake_user_db.get(username)
    if stored_hash is None:
        return False
    return stored_hash == sha256_hex(attempted_password)

print(check_login("alice", "correct-horse-battery-staple"))  # True
print(check_login("alice", "wrong-guess"))                    # False

True
False


## 3.4 Why plain SHA-256 still isn't enough for real password storage

SHA-256 is *fast*, deliberately so, since it's designed for things like verifying
file integrity, where speed is a feature. But that same speed is a liability for
password storage: an attacker with a leaked hash database can try billions of
guesses per second on modern hardware. They can also precompute lookup tables of common
passwords, called **rainbow tables**, especially against short or common
passwords.

Two standard mitigations:

- **Salting**: add a random, unique value (the *salt*) to each password before
  hashing, and store the salt alongside the hash. This defeats precomputed rainbow
  tables, because an attacker would need a separate table per salt.
- **Slow, purpose-built hashing algorithms** (like PBKDF2, bcrypt, scrypt, or
  Argon2) that are deliberately expensive to compute, so brute-forcing many guesses
  becomes impractical even with powerful hardware.

Let's demonstrate salting plus a slow algorithm, using PBKDF2 (built into Python's
standard library):

In [4]:
import os
import hashlib
import binascii

def hash_password(password: str, salt: bytes = None, iterations: int = 200_000):
    if salt is None:
        salt = os.urandom(16)  # unique per user, safe to store alongside the hash
    derived = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations)
    return salt, derived

def verify_password(password: str, salt: bytes, expected: bytes, iterations: int = 200_000) -> bool:
    _, derived = hash_password(password, salt, iterations)
    return derived == expected

salt, stored_hash = hash_password("correct-horse-battery-staple")
print("Salt (store this): ", binascii.hexlify(salt).decode())
print("Hash (store this): ", binascii.hexlify(stored_hash).decode())

print("\nCorrect password accepted:", verify_password("correct-horse-battery-staple", salt, stored_hash))
print("Wrong password rejected:  ", verify_password("wrong-guess", salt, stored_hash))


Salt (store this):  e7438bf5f156d838e424950fa735f6c3
Hash (store this):  e05aa6267c7838aa3d8f82decdda219975f37d347a635381e77a647e4f560939

Correct password accepted: True
Wrong password rejected:   False


Notice two different users with the *same* password now get *different* stored
hashes, because each has a different random salt:


In [5]:
salt1, hash1 = hash_password("hunter2")
salt2, hash2 = hash_password("hunter2")

print("Same password, different salts -> different hashes:")
print(hash1 == hash2)


Same password, different salts -> different hashes:
False


## 3.5 Where this fits into the bigger picture

Hashing shows up again later in this repository in a different role: as a step
inside **digital signatures** (Notebook 4) and inside **JWTs** (Notebooks 5–6).
There, the goal isn't hiding a password. It's producing a compact fingerprint of
a *message* so that a signature only has to cover the fingerprint, not the whole
message.

## Summary

- A hash function turns any input into a fixed-size, one-way fingerprint.
- Small input changes produce completely different hashes (the avalanche effect).
- Systems should store password *hashes*, never plaintext passwords.
- Plain fast hashes (like raw SHA-256) aren't safe for passwords on their own.
  Real systems add a per-user **salt** and use a deliberately **slow** algorithm
  (PBKDF2, bcrypt, scrypt, or Argon2).

**Next:** `04_digital_signatures.ipynb`